## 0. Setup

In [ ]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [ ]:
# general
from pathlib import Path

import geopandas as gpd

# for plotting and coloring
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from tqdm.notebook import tqdm

gpd.options.io_engine = "pyogrio"

In [ ]:
from gridsample.utils import create_gmap_links, save_shapefiles
# from gridsample.mapping import create_interactive_map

In [ ]:
from utils import (
    download_VIDA_rooftops_data_by_s2,
    generate_colormap,
    get_matched_rooftop_centroids_from_s2_file,
    get_s2_cell_ids,
    s2_cell_ids_to_shapes_gdf,
)

In [ ]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data_panel"
RAW_DATA_DIR = DATA_DIR / "01. Raw Data" #symlinked from IFS folder
CLEANED_DATA_DIR = DATA_DIR / "02. Intermediate Outputs" / "v1"
OUTPUT_DATA_DIR = DATA_DIR / "03. Final Outputs" / "v3_full_rerun"
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load MapSolve boundaries

In [ ]:
# get all filepaths that end in `gpkg` inside the RAW_DATA_DIR / 0.1. MapSolve Boundaries
boundaries_dir = RAW_DATA_DIR / "01. MapSolve Boundaries"
gpkg_files_all = list(boundaries_dir.glob("**/*.gpkg"))
gpkg_files_all = [f for f in gpkg_files_all if f.is_file()]
# # drop any with the word "Sub-District" in the filename
# gpkg_files_VTW = [f for f in gpkg_files_all if "Sub-District" not in f.name]
# load all shapes into one gdf
gdf_list = []
for filepath in gpkg_files_all:
    gdf_list.append(gpd.read_file(filepath))
gdf = pd.concat(gdf_list, ignore_index=True).to_crs(4326)

In [ ]:
gdf = gdf.drop_duplicates()

### Fix ad-hoc issues (see quality check notebook)

In [ ]:
# chittoor
gdf.loc[gdf["TV_C"] == 803014, ["SubDist_N", "SubDist_C"]] = ["Tirupati (Urban)", 5383]
# patna
gdf.loc[(gdf["TV_C"] == 801373) & (gdf["Ward_C"] == "3"), "PCA_ID"] = "801373-3"
# MDDS codes
gdf.loc[gdf["TV_C"] == 519923, ["TV_C", "Ward_C", "PCA_ID"]] = [802596, 18, "802596-18"]
gdf.loc[gdf["TV_C"] == 574077, ["TV_C", "Ward_C", "PCA_ID"]] = [802918, 157, "802918-157"]

In [ ]:
gdf.plot(column="State_N", legend=True, figsize=(10, 10), cmap="tab20")

## 2. Load sampled wards data

In [ ]:
# load the merged wards data
sample_df = pd.read_csv(
    CLEANED_DATA_DIR
    / "00. Merge and Quality Checks"
    / "Panel Wards with Quality Checks.csv"
)

In [ ]:
sample_df

### 2.1 Rename and clean both datasets

In [ ]:
rename_dict = {
    "UID": "UID",
    "PCA_ID": "PCA_ID",
    "State_C": "State Code",
    "State_N": "State Name",
    "Dist_C": "District Code",
    "Dist_N": "District Name",
    "SubDist_C": "Subdistrict Code",
    "SubDist_N": "Subdistrict Name",
    "TV_C": "TV Code",
    "TV_N": "TV Name",
    "Ward_C": "Ward Code",
    "TOT_P": "Total Population",
}

gdf = gdf.rename(columns=rename_dict)

In [ ]:
rename_dict = {
    "State": "State Code",
    "State_Name": "State Name",
    "District": "District Code",
    "District_Name": "District Name",
    "Subdistrict": "Subdistrict Code",
    "Subd_Name": "Subdistrict Name",
    "TownVillage": "TV Code",
    "UrbanWardVillage": "Ward Code",
    "WardVillage_Name": "Ward/Village Name",
    "TRU": "Urban/Rural",
    "WardVillage_Pop": "Ward Population",
    "Subd_Pop": "Subdistrict Population",
    "State_Pop": "State Population",
    "WardVillageID": "Complete ID",
}
sample_df = sample_df.rename(columns=rename_dict)

# make State Name heading case instead of all caps
sample_df["State Name"] = sample_df["State Name"].str.title()
sample_df.loc[sample_df["State Name"] == "Nct Of Delhi", "State Name"] = "NCT of Delhi"

In [ ]:
# make relevant codes into floats for both datasets
code_columns = [
    "State Code",
    "District Code",
    "Subdistrict Code",
    "TV Code",
    "Ward Code",
]
for col in code_columns:
    sample_df[col] = sample_df[col].astype(float)
    gdf[col] = gdf[col].astype(float)

In [ ]:
sample_df[sample_df["District Name"] == "Durg"]

## 3. Filter MapSolve boundaries to sampled areas

### 3.2 Drop unnecessary rows 

#### Drop rows with no MapSolve shapes

In [ ]:
sample_df[sample_df["PSU Type"] == "none"]

In [ ]:
filtered_df = sample_df[sample_df["PSU Type"] != "none"]

In [ ]:
filtered_df

In [ ]:
# check bad rows, leave as is though
filtered_df[filtered_df["Delivery State"].str.contains("BAD")]

#### (not needed, drop duplicated on PSU ID solves this) Drop duplicate rows if PSU is TV or Subdistrict

In [ ]:
# # GOAL: if we have multiple rows for the same subdistrict or village PSU, keep only the first one. We still know how many to sample 
# # because we have the "Ward Count" column.

# # filter PSU type
# temp_tv_filtered_df = filtered_df[filtered_df["PSU Type"] == "town_village"]
# # filter to only those rows that are duplicated in the code column
# temp_tv_filtered_df_dup = temp_tv_filtered_df[
#     temp_tv_filtered_df.duplicated(subset=["TV Code"], keep=False)
# ]
# # get the indices of the rows to drop (i.e. all but the first occurrence of each duplicated code)
# tv_indices_to_drop = temp_tv_filtered_df_dup[
#     ~temp_tv_filtered_df_dup.duplicated(subset=["TV Code"], keep="first")
# ].index
# # drop those rows from the filtered_df
# filtered_df = filtered_df[~filtered_df.index.isin(tv_indices_to_drop)]


# # Repeat for subdistricts
# temp_subd_filtered_df = filtered_df[filtered_df["PSU Type"] == "subdistrict"]
# temp_subd_filtered_df_dup = temp_subd_filtered_df[
#     temp_subd_filtered_df.duplicated(subset=["Subdistrict Code"], keep=False)
# ]
# subd_indices_to_drop = temp_subd_filtered_df_dup[
#     ~temp_subd_filtered_df_dup.duplicated(subset=["Subdistrict Code"], keep="first")
# ].index
# filtered_df = filtered_df[~filtered_df.index.isin(subd_indices_to_drop)]

# filtered_df.value_counts("PSU Type")

In [ ]:
# filtered_df[filtered_df["District Name"] == "Durg"]

#### Duplicated wards

In [ ]:
filtered_df[filtered_df["PCA_ID"].duplicated(keep=False)]

In [ ]:
print(f"Number of rows before dropping PCA_ID duplicates: {len(filtered_df)}")
print(
    f"Number of rows after dropping PCA_ID duplicates: {len(filtered_df.drop_duplicates(subset=['PCA_ID'], keep='first'))}"
)

In [ ]:
filtered_df = filtered_df.drop_duplicates(subset=["PCA_ID"], keep="first")

#### Check for duplicated `PSU ID`

In [ ]:
filtered_df[filtered_df.duplicated(subset=["PSU ID"], keep=False)]

In [ ]:
filtered_df = filtered_df.drop_duplicates(subset=["PSU ID"], keep="first")
len(filtered_df) == filtered_df["PSU ID"].nunique()

In [ ]:
filtered_df[filtered_df["District Name"] == "Durg"]

## 4. Merge shapes at the respective PSU level

### Wards

In [ ]:
# PCA IDs to match (TVCode-WardCode)
pca_ids_to_match = filtered_df.loc[filtered_df["PSU Type"] == "ward", "PCA_ID"].unique()
len(pca_ids_to_match)

In [ ]:
wards_gdf = gpd.GeoDataFrame(
    filtered_df[filtered_df["PSU Type"] == "ward"].merge(
        gdf[gdf["PCA_ID"].isin(pca_ids_to_match)],
        on=["PCA_ID"],
        how="left",
        suffixes=("", "_MapSolve"),
    ),
    geometry="geometry",
)
wards_gdf.shape

### Town/Village

In [ ]:
# TV codes to match
tv_codes_to_match = filtered_df.loc[
    filtered_df["PSU Type"] == "town_village", "TV Code"
].unique()
len(tv_codes_to_match)

In [ ]:
tv_gdf = gpd.GeoDataFrame(
    filtered_df[filtered_df["PSU Type"] == "town_village"].merge(
        gdf[gdf["TV Code"].isin(tv_codes_to_match) & gdf["Ward Code"].isna()],
        on=["TV Code"],
        how="left",
        suffixes=("", "_MapSolve"),
    )
)
tv_gdf.shape

In [ ]:
# cut out the ward geometries from the TV geometries so we don't double sample those areas
trimmed_tv_geoms = tv_gdf.difference(wards_gdf.geometry.unary_union)
trimmed_tv_gdf = gpd.GeoDataFrame(
    tv_gdf.drop(columns="geometry").assign(geometry=trimmed_tv_geoms),
    crs=tv_gdf.crs,
)

### Subdistrict

In [ ]:
# subdistrict codes to match
subdistrict_codes_to_match = filtered_df.loc[
    filtered_df["PSU Type"] == "subdistrict", "Subdistrict Code"
].unique()
len(subdistrict_codes_to_match)

In [ ]:
subdistrict_gdf = gpd.GeoDataFrame(
    filtered_df[filtered_df["PSU Type"] == "subdistrict"].merge(
        gdf[
            gdf["Subdistrict Code"].isin(subdistrict_codes_to_match)
            & (gdf["TV Code"].isna())
            & (gdf["Ward Code"].isna())
        ],
        on=["Subdistrict Code"],
        how="left",
        suffixes=("", "_MapSolve"),
    )
)

In [ ]:
# cut out the TV and Ward geometries from the subdistrict geometries so we don't double sample those areas
combined_tv_and_ward_union_shape = wards_gdf.geometry.unary_union.union(
    trimmed_tv_gdf.geometry.unary_union
)
trimmed_subdistrict_geoms = subdistrict_gdf.difference(combined_tv_and_ward_union_shape)
trimmed_subdistrict_gdf = gpd.GeoDataFrame(
    subdistrict_gdf.drop(columns="geometry").assign(geometry=trimmed_subdistrict_geoms),
    crs=subdistrict_gdf.crs,
)

### Combine

In [ ]:
# combine all three GeoDataFrames into one
combined_gdf = gpd.GeoDataFrame(
    pd.concat(
        [
            wards_gdf,
            trimmed_tv_gdf,
            trimmed_subdistrict_gdf,
        ],
        ignore_index=True,
    )
)
combined_gdf = combined_gdf.sort_values(
    by=["State Code", "District Code", "Subdistrict Code", "TV Code", "Ward Code"]
).reset_index(drop=True)

In [ ]:
combined_gdf.plot(column="State Name")

### Drop rows duplicated `PSU ID` (could happen)

In [ ]:
duplicated_psu_id_gdf = combined_gdf[combined_gdf.duplicated(subset=["PSU ID"], keep=False)]
duplicated_psu_id_gdf

In [ ]:
duplicated_psu_id_gdf.groupby("PSU ID").plot(alpha=0.5, column="PSU ID", legend=True)

In [ ]:
combined_gdf = combined_gdf.drop_duplicates(subset=["PSU ID"], keep="first")
len(combined_gdf) == combined_gdf["PSU ID"].nunique()

In [ ]:
combined_gdf["UID"] = combined_gdf["UID"].astype(str)

In [ ]:
len(combined_gdf) == combined_gdf["PSU ID"].nunique()

In [ ]:
save_shapefiles(
    combined_gdf,
    OUTPUT_DATA_DIR / "Sampled PSUs",
    "all_sampled_PSUs",
    ["csv", "parquet", "kml"],
)

## 5. Download rooftops

#### Identify S2 cell IDs

In [ ]:
s2_cell_ids = get_s2_cell_ids(combined_gdf)
len(s2_cell_ids)

#### Check if identified cells cover all areas of interest

In [ ]:
s2_cells_gdf = s2_cell_ids_to_shapes_gdf(s2_cell_ids)

In [ ]:
# Does the S2 cell cover the entire area of the boundaries?
uncovered_area = combined_gdf.unary_union.difference(s2_cells_gdf.unary_union).area
print(f"{uncovered_area} square degrees area not covered by an S2 cell")

In [ ]:
# Plot the S2 cells and the boundary
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
combined_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
s2_cells_gdf.plot(ax=ax, facecolor="none", edgecolor="red", alpha=0.7)
plt.title("S2 Cells Level 6 Coverage")
plt.tight_layout()
plt.show()

In [ ]:
s2_cells_gdf_w_state = (
    s2_cells_gdf.sjoin(
        combined_gdf[["State Name", "geometry"]], how="inner", predicate="intersects"
    )
    .drop(columns="index_right")
    .drop_duplicates()
)

In [ ]:
# note: this will have duplicate s2 cell rows with different state names if the s2 cell overlaps multiple states
# this is expected and is required for the next steps logic to work correctly
s2_cells_gdf_w_state

#### Download the S2 cells

In [ ]:
download_VIDA_rooftops_data_by_s2(s2_cell_ids, "IND", RAW_DATA_DIR / "02. Rooftop Data")

## 6. Load rooftops and match to areas

In [ ]:
state_names = combined_gdf["State Name"].sort_values().unique()
# state_names = ["Jharkhand", "Odisha", "Chhattisgarh"]

In [ ]:
# for state_name in tqdm(state_names):
#     print(f"Processing state: {state_name}")

#     # Filter the s2 cells and rooftops gdf to the current state
#     s2_cell_ids = set(
#         s2_cells_gdf_w_state[s2_cells_gdf_w_state["State Name"] == state_name][
#             "s2_cell_id"
#         ]
#     )
#     print(
#         f"Number of S2 cells that overlap our shapes in {state_name}: {len(s2_cell_ids)}"
#     )
#     gdf_subset = combined_gdf[combined_gdf["State Name"] == state_name]

#     # Get matched rooftops for each S2 cell in the state
#     matched_rooftop_centroids_gdf_list = []
#     for s2_cell_id in tqdm(s2_cell_ids):
#         matched_rooftop_centroids_gdf = get_matched_rooftop_centroids_from_s2_file(
#             s2_file_dir=RAW_DATA_DIR / "02. Rooftop Data",
#             s2_cell_id=s2_cell_id,
#             boundaries_gdf=gdf_subset,
#         )
#         matched_rooftop_centroids_gdf_list.append(matched_rooftop_centroids_gdf)
#     matched_rooftop_centroids_gdf = pd.concat(
#         matched_rooftop_centroids_gdf_list, ignore_index=True
#     )
#     matched_rooftop_centroids_gdf["State Name"] = state_name

#     # Save the matched rooftops data
#     save_shapefiles(
#         matched_rooftop_centroids_gdf,
#         CLEANED_DATA_DIR / "01. Matched Rooftop Data" / f"{state_name}",
#         "matched_rooftops",
#         ["parquet"],
#     )

In [ ]:
# ax = matched_rooftop_centroids_gdf.sample(1000).plot(
#     cmap=ListedColormap(generate_colormap(len(matched_rooftop_centroids_gdf))),
# )
# gdf_subset.plot(ax=ax, color="none", edgecolor="black", linewidth=0.5)

## 7. Load matched rooftops

In [ ]:
matched_rooftop_dir = CLEANED_DATA_DIR / "01. Matched Rooftop Data"
all_filepaths = list(matched_rooftop_dir.glob("**/*.parquet"))
all_filepaths = [f for f in all_filepaths if f.is_file()]

# filter to those that have selected_states in the name
all_filepaths = [
    f for f in all_filepaths if any(state in f.parent.name for state in state_names)
]

# load all shapes into one gdf
matched_rooftops_gdf_list = []
for filepath in tqdm(all_filepaths):
    matched_rooftops_gdf_list.append(gpd.read_parquet(filepath))
matched_rooftops_gdf = gpd.GeoDataFrame(
    pd.concat(matched_rooftops_gdf_list, ignore_index=True)
).to_crs(4326)

In [ ]:
len(matched_rooftops_gdf)

In [ ]:
no_rooftop_PSU_IDs = set(combined_gdf[combined_gdf["State Name"].isin(state_names)]["PSU ID"].unique()).difference(
    set(matched_rooftops_gdf["PSU ID"].unique())
)
no_rooftop_PSU_gdf = combined_gdf[combined_gdf["PSU ID"].isin(no_rooftop_PSU_IDs)]
no_rooftop_PSU_gdf

In [ ]:
save_shapefiles(
    no_rooftop_PSU_gdf,
    OUTPUT_DATA_DIR / "Sampled PSUs",
    "PSUs_with_no_rooftops",
    ["csv", "kml"],
)

## 8. Sample rooftops

In [ ]:
# # filter just to problematic districts
# district_names = ["Ranchi", "Khordha", "Durg", "Raipur"]
# matched_rooftops_gdf = matched_rooftops_gdf[matched_rooftops_gdf["District Name"].isin(district_names)]

In [ ]:
# Define the base number of rooftops per ward
ROOFTOPS_PER_WARD = 75

# Sample rooftops, multiplying by Ward Count for each PSU.
sampled_rooftops = matched_rooftops_gdf.groupby("PSU ID", group_keys=False).apply(
    lambda x: x.sample(
        n=min(ROOFTOPS_PER_WARD * int(x["Ward Count"].iloc[0]), x.shape[0]),
        random_state=42,
    )
)

In [ ]:
print("Length of matched_rooftops_gdf:", len(matched_rooftops_gdf))
print("Length of sampled rooftops:", len(sampled_rooftops))

In [ ]:
# Check if Ward Count is correctly influencing sample sizes
TEMP_ward_count_df = matched_rooftops_gdf[["PSU ID", "Ward Count"]].drop_duplicates()
TEMP_ward_count_df["Expected Rooftop Count"] = TEMP_ward_count_df["Ward Count"] * ROOFTOPS_PER_WARD
TEMP_sampled_counts = (
    sampled_rooftops.groupby("PSU ID").size()
).reset_index(name="Sampled Rooftop Count")

# Merge the two dataframes
TEMP_check_df = TEMP_ward_count_df.merge(TEMP_sampled_counts, on="PSU ID")
TEMP_check_df["Rooftop Count Difference"] = (
    TEMP_check_df["Expected Rooftop Count"] - TEMP_check_df["Sampled Rooftop Count"]
)
TEMP_check_df[TEMP_check_df["Rooftop Count Difference"] != 0]

In [ ]:
sampled_rooftops.plot(
    figsize=(8, 8),
    column="State Name",
    cmap="tab20",
    edgecolor="black",
    linewidth=0.5,
    legend=True,
)

### Add sample-level rooftop numbering ID columns

In [ ]:
# Rooftop number within each state
sampled_rooftops["Rooftop State ID"] = (
    sampled_rooftops.groupby("State Name").cumcount() + 1
)

# Rooftop number within each PSU ID
sampled_rooftops["Rooftop PSU ID Numeric"] = sampled_rooftops.groupby("PSU ID").cumcount() + 1
# add prefix of "PIN "  to the Rooftop PSU ID
sampled_rooftops["Rooftop PSU ID"] = "PIN " + sampled_rooftops["Rooftop PSU ID Numeric"].astype(
    str
)

# Rooftop unique ID
sampled_rooftops["Rooftop Unique ID"] = sampled_rooftops.apply(
    lambda row: f"STATE_{row['Rooftop State ID']}_PSU_ID_{row['PSU ID']}_ROOFTOP_{row['Rooftop PSU ID']}",
    axis=1,
)

### Add gmap link

In [ ]:
sampled_rooftops["latitude_original"] = sampled_rooftops.geometry.y
sampled_rooftops["longitude_original"] = sampled_rooftops.geometry.x
sampled_rooftops["gmap_link_original"] = create_gmap_links(
    df=sampled_rooftops,
    lat_name="latitude_original",
    lon_name="longitude_original",
)

### Select only useful columns

**Required columns:**
- PSU info
    - Unique ID across all rooftops
    - Rooftop state ID, #
    - Rooftop PSU ID, #

    - PSU Unit: Ward, TV, Subdistrict
    - PSU sample size

- geospatial info
    - google maps link
    - coordinates
    - geometry

- Admin location info
    - State code and name
    - District code and Name
    - Subdistrict code and name
    - TV code and name
    - Ward code and name

In [ ]:
chosen_cols = [
    ## IDs
    "Rooftop State ID",
    "Rooftop PSU ID",
    "Rooftop PSU ID Numeric",
    "Rooftop Unique ID",
    ## Geospatial data
    "geometry",
    "latitude_original",
    "longitude_original",
    "gmap_link_original",
    ## PSU info
    "PSU ID",
    "PSU Type",
    "Ward Count",
    ## Location info
    "State Code",
    "State Name",
    "State Changed",
    "District Code",
    "District Name",
    "Subdistrict Code",
    "Subdistrict Name",
    "TV Code",
    "TV Name",  # (from MapSolve)
    "Ward Code",
    "Ward/Village Name",
    "Urban/Rural",
    "PCA_ID",  # combined TVCode-WardCode
    "Ward Population",
    "Subdistrict Population",
    "State Population",
    # "Complete ID",
    ## Admin information
    "Included in Panel",
    "Ward Boundary Available with MapSolve",
    # "State Shared by MapSolve",
    "Ward Boundary Given",
    "TV Boundary Given",
    "SubDistrict Boundary Given",
    "Delivery State",
    # "UID",
    # "s2_rooftop_id",
    ## MapSolve location info
    "State Code_MapSolve",
    "State Name_MapSolve",
    "District Code_MapSolve",
    "District Name_MapSolve",
    "Subdistrict Code_MapSolve",
    "Subdistrict Name_MapSolve",
    "TV Code_MapSolve",
    # "TV Name",
    "Ward Code_MapSolve",
    "PCA_ID_MapSolve",
    "Total Population",
    # ## rooftop info
    # "boundary_id",
    # "bf_source",
    # "confidence",
    # "area_in_meters",
    # "s2_id",
    # "country_iso",
    # "geohash",
    # "bbox",
]

In [ ]:
sampled_rooftops_organised_gdf = sampled_rooftops[chosen_cols]

In [ ]:
sampled_rooftops_organised_gdf.rename(
    columns={
        "TV Name": "TV Name_MapSolve",
        "PSU ID": "PSU ID",
        "Total Population": "PSU Total Population_MapSolve",
    },
    inplace=True,
)

In [ ]:
# set Ward Codes of 0.0 to NaN
sampled_rooftops_organised_gdf.loc[
    sampled_rooftops_organised_gdf["Ward Code"] == 0.0, "Ward Code"
] = np.nan

In [ ]:
sampled_rooftops_organised_gdf

### Save sampled data (original rooftop pins)

In [ ]:
sampled_rooftops_organised_gdf["Subdistrict Population"] = sampled_rooftops_organised_gdf["Subdistrict Population"].astype(str)
sampled_rooftops_organised_gdf["State Population"] = sampled_rooftops_organised_gdf["State Population"].astype(str)

In [ ]:
save_shapefiles(
    sampled_rooftops_organised_gdf,
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_centroids_original",
    ["csv", "parquet"],
)

save_shapefiles(
    sampled_rooftops_organised_gdf.drop(columns=["gmap_link_original"]),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_centroids_original",
    ["kml"],
)

## 9. Snap points to road

In [ ]:
import yaml
from shapely import Point

from utils import get_nearest_points_on_road_batch, get_nearest_points_on_road_batch_parallel

In [ ]:
# load API key
with open("../secrets/api_keys.yaml", "r") as f:
    config = yaml.safe_load(f)
    api_key = config["GOOGLE_ROADS_API_KEY"]

In [ ]:
get_nearest_points_on_road_batch([Point(77.11432151622034, 28.677391409999522)], api_key)

In [ ]:
# test
get_nearest_points_on_road_batch(sampled_rooftops_organised_gdf.geometry.iloc[:5], api_key)

In [ ]:
get_nearest_points_on_road_batch_parallel(sampled_rooftops_organised_gdf.iloc[:150], api_key)

#### Snap points to road

In [ ]:
snapped_points_series = get_nearest_points_on_road_batch_parallel(
    sampled_rooftops_organised_gdf, api_key, max_workers=12
)
# took 1 second for 1,600 points (Panel)
# 30s for all 53,000 points 

In [ ]:
sampled_rooftops_snapped_gdf = sampled_rooftops_organised_gdf.copy()
sampled_rooftops_snapped_gdf["geometry_snapped"] = sampled_rooftops_snapped_gdf.index.map(snapped_points_series)

In [ ]:
# Make new Geometry Type column which has values "Original" or "Snapped to Road"
sampled_rooftops_snapped_gdf["Geometry Type"] = (
    sampled_rooftops_snapped_gdf["geometry_snapped"]
    .notna()
    .replace({True: "Snapped to Road", False: "Original"})
)
sampled_rooftops_snapped_gdf["Geometry Type"].value_counts()

#### Replace geometry to snapped one (missing filled in with original)

In [ ]:
# backup the original geometry
sampled_rooftops_snapped_gdf["geometry_original"] = sampled_rooftops_snapped_gdf[
    "geometry"
]
# replace the original geometry with the snapped geometry
sampled_rooftops_snapped_gdf["geometry"] = sampled_rooftops_snapped_gdf[
    "geometry_snapped"
]
# drop the snapped geometry column
sampled_rooftops_snapped_gdf = sampled_rooftops_snapped_gdf.drop(
    columns=["geometry_snapped"]
)
# fill in NaN values in the snapped geometry with the original geometry
sampled_rooftops_snapped_gdf["geometry"] = sampled_rooftops_snapped_gdf[
    "geometry"
].fillna(sampled_rooftops_snapped_gdf["geometry_original"])

In [ ]:
sampled_rooftops_snapped_gdf["geometry"].isna().sum()

#### Update lat, lon, gmap_link

In [ ]:
sampled_rooftops_snapped_gdf["latitude"] = list(sampled_rooftops_snapped_gdf.geometry.y)
sampled_rooftops_snapped_gdf["longitude"] = list(
    sampled_rooftops_snapped_gdf.geometry.x
)
sampled_rooftops_snapped_gdf["gmap_link"] = create_gmap_links(
    df=sampled_rooftops_snapped_gdf,
    lat_name="latitude",
    lon_name="longitude",
)

#### Reorganise

In [ ]:
sampled_rooftops_snapped_gdf = sampled_rooftops_snapped_gdf[
    [
        "Rooftop State ID",
        "Rooftop PSU ID",
        "Rooftop PSU ID Numeric",
        "Rooftop Unique ID",
        # new columns start
        "Geometry Type",
        "geometry",
        "latitude",
        "longitude",
        "gmap_link",
        # new columns end
        "geometry_original",
        "latitude_original",
        "longitude_original",
        "gmap_link_original",
        "PSU ID",
        "PSU Type",
        "Ward Count",
        "State Code",
        "State Name",
        "State Changed",
        "District Code",
        "District Name",
        "Subdistrict Code",
        "Subdistrict Name",
        "TV Code",
        "TV Name_MapSolve",
        "Ward Code",
        "Ward/Village Name",
        "Urban/Rural",
        "PCA_ID",
        "Ward Population",
        "Subdistrict Population",
        "State Population",
        "Included in Panel",
        "Ward Boundary Available with MapSolve",
        "Ward Boundary Given",
        "TV Boundary Given",
        "SubDistrict Boundary Given",
        "Delivery State",
        "State Code_MapSolve",
        "State Name_MapSolve",
        "District Code_MapSolve",
        "District Name_MapSolve",
        "Subdistrict Code_MapSolve",
        "Subdistrict Name_MapSolve",
        "TV Code_MapSolve",
        "Ward Code_MapSolve",
        "PCA_ID_MapSolve",
        "PSU Total Population_MapSolve",
    ]
]

#### Make lines between original and snapped points

In [ ]:
from shapely.geometry import LineString

In [ ]:
sampled_rooftops_snapped_gdf["geometry_line"] = sampled_rooftops_snapped_gdf.apply(
    lambda row: LineString([row["geometry_original"], row["geometry"]]), axis=1
)

In [ ]:
sampled_rooftops_snapped_gdf["geometry_line"].isna().sum()

#### Save new files: snapped points, snapped lines

In [ ]:
# Save CSV and parquet
save_shapefiles(
    sampled_rooftops_snapped_gdf.drop(
        columns=[
            "geometry_original",
            "geometry_line",
        ]
    ),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_snapped_points",
    ["csv", "parquet"],
)

In [ ]:
# Save KML
save_shapefiles(
    sampled_rooftops_snapped_gdf.drop(
        columns=[
            "geometry_original",
            "geometry_line",
            # bad cols for KML
            "gmap_link_original",
            "gmap_link"
        ]
    ),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_snapped_points",
    ["kml"],
)

In [ ]:
# Save lines
sampled_rooftops_line_gdf = sampled_rooftops_snapped_gdf.copy()
sampled_rooftops_line_gdf["geometry"] = sampled_rooftops_line_gdf["geometry_line"]
sampled_rooftops_line_gdf = sampled_rooftops_line_gdf.drop(
    columns=["geometry_original", "geometry_line"]
)

In [ ]:
save_shapefiles(
    sampled_rooftops_line_gdf.drop(
        # drop kml unfriendly columns
        columns=["gmap_link_original", "gmap_link"]
    ),
    OUTPUT_DATA_DIR / "01. Sampled Rooftop Data",
    "sampled_rooftops_snapped_lines",
    ["parquet", "kml"],
)

## Save per state

In [ ]:
for state in tqdm(sampled_rooftops_organised_gdf["State Name"].unique()):
    state_output_folder = OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / state

    # save original points
    selected_state_original_gdf = sampled_rooftops_organised_gdf[
        sampled_rooftops_organised_gdf["State Name"] == state
    ]

    save_shapefiles(
        selected_state_original_gdf,
        state_output_folder,
        f"{state}_sampled_rooftops_centroids_original",
        ["csv", "parquet"],
    )
    save_shapefiles(
        selected_state_original_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original"]
        ),
        state_output_folder,
        f"{state}_sampled_rooftops_centroids_original",
        ["kml"],
    )

    # Save snapped points
    selected_state_snapped_gdf = sampled_rooftops_snapped_gdf[
        sampled_rooftops_snapped_gdf["State Name"] == state
    ].drop(
        columns=[
            "geometry_original",
            "geometry_line",
        ]
    )
    save_shapefiles(
        selected_state_snapped_gdf,
        state_output_folder,
        f"{state}_sampled_rooftops_snapped_points",
        ["csv", "parquet"],
    )
    save_shapefiles(
        selected_state_snapped_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original", "gmap_link"]
        ),
        state_output_folder,
        f"{state}_sampled_rooftops_snapped_points",
        ["kml"],
    )

    # Save lines
    selected_state_sampled_rooftops_line_gdf = sampled_rooftops_line_gdf[
        sampled_rooftops_line_gdf["State Name"] == state
    ]
    save_shapefiles(
        selected_state_sampled_rooftops_line_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original", "gmap_link"]
        ),
        state_output_folder,
        f"{state}_sampled_rooftops_snapped_lines",
        ["parquet", "kml"],
    )

### Save slices per state

In [ ]:
from math import ceil
SLICE_SIZE = 25
N_SLICES = ceil(ROOFTOPS_PER_WARD / SLICE_SIZE)

print(f"ROOFTOPS_PER_WARD: {ROOFTOPS_PER_WARD}")
print(f"SLICE_SIZE: {SLICE_SIZE}")
print(f"N_SLICES: {N_SLICES}")
print(f"N_SLICES * SLICE_SIZE: {N_SLICES * SLICE_SIZE}")

In [ ]:
sampled_rooftops_organised_gdf["State Name"].unique()

In [ ]:
for state in tqdm(sampled_rooftops_organised_gdf["State Name"].unique()):
    state_gdf = sampled_rooftops_organised_gdf[
        sampled_rooftops_organised_gdf["State Name"] == state
    ]
    for slice_idx in range(N_SLICES):

        original_points_slice_rows = []
        snapped_points_slice_rows = []
        line_slice_rows = []

        for psu_id, psu_df in state_gdf.groupby("PSU ID"):
            # get counts and indices from the original dataframe (also applies to the other two)
            ward_count = int(psu_df["Ward Count"].iloc[0])
            start_idx = slice_idx * SLICE_SIZE * ward_count
            end_idx = start_idx + (SLICE_SIZE * ward_count)

            # original points
            psu_original_slice = sampled_rooftops_organised_gdf[
                (sampled_rooftops_organised_gdf["State Name"] == state)
                & (sampled_rooftops_organised_gdf["PSU ID"] == psu_id)
            ].iloc[start_idx:end_idx]
            original_points_slice_rows.append(psu_original_slice)

            # snapped points
            psu_snapped_slice = sampled_rooftops_snapped_gdf[
                (sampled_rooftops_snapped_gdf["State Name"] == state)
                & (sampled_rooftops_snapped_gdf["PSU ID"] == psu_id)
            ].iloc[start_idx:end_idx]
            snapped_points_slice_rows.append(psu_snapped_slice)

            # lines
            psu_line_slice = sampled_rooftops_line_gdf[
                (sampled_rooftops_line_gdf["State Name"] == state)
                & (sampled_rooftops_line_gdf["PSU ID"] == psu_id)
            ].iloc[start_idx:end_idx]
            line_slice_rows.append(psu_line_slice)


        # Concatenate all PSU slices for this state and slice
        state_slice_original_gdf = pd.concat(original_points_slice_rows, ignore_index=True)
        state_slice_snapped_gdf = pd.concat(snapped_points_slice_rows, ignore_index=True)
        state_slice_line_gdf = pd.concat(line_slice_rows, ignore_index=True)

        # set folder
        state_slice_output_folder = (
            OUTPUT_DATA_DIR
            / "01. Sampled Rooftop Data"
            / state
            / f"slice_{slice_idx}"
        )

        # Save original points
        save_shapefiles(
            state_slice_original_gdf,
            state_slice_output_folder,
            f"{state}_sampled_rooftops_centroids_original_{slice_idx}",
            ["csv", "parquet"],
        )
        save_shapefiles(
            state_slice_original_gdf.drop(columns=["gmap_link_original"]),
            state_slice_output_folder,
            f"{state}_sampled_rooftops_centroids_original_{slice_idx}",
            ["kml"],
        )

        # Save snapped points
        save_shapefiles(
            state_slice_snapped_gdf,
            state_slice_output_folder,
            f"{state}_sampled_rooftops_snapped_points_{slice_idx}",
            ["csv", "parquet"],
        )
        save_shapefiles(
            state_slice_snapped_gdf.drop(
                # drop kml unfriendly columns
                columns=["gmap_link_original", "gmap_link", "geometry_line", "geometry_original"]
            ),
            state_slice_output_folder,
            f"{state}_sampled_rooftops_snapped_points_{slice_idx}",
            ["kml"],
        )

        # Save lines
        save_shapefiles(
            state_slice_line_gdf.drop(
                # drop kml unfriendly columns
                columns=["gmap_link_original", "gmap_link"]
            ),
            state_slice_output_folder,
            f"{state}_sampled_rooftops_snapped_lines_{slice_idx}",
            ["parquet", "kml"],
        )

## Save per-PSU

In [ ]:
for psu_id in tqdm(sampled_rooftops_organised_gdf["PSU ID"].unique()):

    # save original points
    selected_psu_original_gdf = sampled_rooftops_organised_gdf[
        sampled_rooftops_organised_gdf["PSU ID"] == psu_id
    ]

    # for folder name
    first_row = selected_psu_original_gdf.iloc[0]
    psu_type = first_row["PSU Type"]
    state_name = first_row["State Name"].strip().replace(" ", "_")
    district_name = first_row["District Name"].strip().replace(" ", "_")
    subdistrict_name = first_row["Subdistrict Name"].strip().replace(" ", "_")
    tv_code = int(first_row["TV Code"]) if pd.notna(first_row["TV Code"]) else None
    ward_code = int(first_row["Ward Code"]) if pd.notna(first_row["Ward Code"]) else None

    if psu_type == "ward":
        foldername = f"{state_name}_DIST_{district_name}_SUBDIST_{subdistrict_name}_TV_{tv_code}_WARD_{ward_code}_PSUID_{psu_id}"
    elif psu_type == "town_village":
        foldername = f"{state_name}_DIST_{district_name}_SUBDIST_{subdistrict_name}_TV_{tv_code}_PSUID_{psu_id}"
    elif psu_type == "subdistrict":
        foldername = f"{state_name}_DIST_{district_name}_SUBDIST_{subdistrict_name}_PSUID_{psu_id}"
    else:
        raise ValueError(f"Unknown PSU Type: {psu_type}")
    psu_output_folder = OUTPUT_DATA_DIR / "01. Sampled Rooftop Data" / "PSU-Level" / foldername
    psu_output_folder.mkdir(parents=True, exist_ok=True)

    save_shapefiles(
        selected_psu_original_gdf,
        psu_output_folder,
        f"{psu_id}_sampled_rooftops_centroids_original",
        ["csv", "parquet"],
    )
    save_shapefiles(
        selected_psu_original_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original"]
        ),
        psu_output_folder,
        f"{psu_id}_sampled_rooftops_centroids_original",
        ["kml"],
    )

    # Save snapped points
    selected_psu_snapped_gdf = sampled_rooftops_snapped_gdf[
        sampled_rooftops_snapped_gdf["PSU ID"] == psu_id
    ].drop(
        columns=[
            "geometry_original",
            "geometry_line",
        ]
    )
    save_shapefiles(
        selected_psu_snapped_gdf,
        psu_output_folder,
        f"{psu_id}_sampled_rooftops_snapped_points",
        ["csv", "parquet"],
    )
    save_shapefiles(
        selected_psu_snapped_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original", "gmap_link"]
        ),
        psu_output_folder,
        f"{psu_id}_sampled_rooftops_snapped_points",
        ["kml"],
    )

    # Save lines
    selected_psu_sampled_rooftops_line_gdf = sampled_rooftops_line_gdf[
        sampled_rooftops_line_gdf["PSU ID"] == psu_id
    ]
    save_shapefiles(
        selected_psu_sampled_rooftops_line_gdf.drop(
            # drop kml unfriendly columns
            columns=["gmap_link_original", "gmap_link"]
        ),
        psu_output_folder,
        f"{psu_id}_sampled_rooftops_snapped_lines",
        ["parquet", "kml"],
    )